# Khasi Companion — Step 1: Inspect & Prepare Shared Datasets
Welcome! This notebook is designed so that anyone on the team—**even with zero AI or Machine Learning background**—can inspect, audit, and clean the dataset folder shared by teammates.

### Core Rules We Follow:
1. **Strict Separation**: Language datasets (System A) are separate from Tourism facts (System B).
2. **Zero-Hallucination**: Never invent translations or facts.
3. **Verification Discipline**: Check for empty rows, duplicates, and column formats before any model training.

### Step 1: Connect to Google Drive (If using Google Colab)
If running locally on your computer in VS Code or Jupyter, you can skip this cell!

In [ ]:
# If running in Google Colab, uncomment and run:
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_PATH = '/content/drive/MyDrive/khasi-companion/data/raw_datasets'

# If running locally:
from pathlib import Path
DATASET_PATH = Path('data/raw_datasets')
print(f"Scanning directory: {DATASET_PATH}")

### Step 2: Automated Dataset Inspector
Run this cell to scan all files in your dataset folder. It checks row counts, columns, null values, and exact duplicates.

In [ ]:
import os
from pathlib import Path
import pandas as pd

def inspect_folder(folder_path):
    base = Path(folder_path)
    if not base.exists():
        print(f"Folder '{folder_path}' does not exist yet. Please create it!")
        return
        
    files = [f for f in base.rglob('*') if f.is_file() and not f.name.startswith('.')]
    print(f"=== Found {len(files)} file(s) in {folder_path} ===\n")
    
    for file_path in files:
        ext = file_path.suffix.lower()
        print(f"--------------------------------------------------")
        print(f"File: {file_path.name} ({file_path.stat().st_size / 1024:.1f} KB)")
        try:
            if ext == '.csv':
                df = pd.read_csv(file_path)
            elif ext == '.tsv':
                df = pd.read_csv(file_path, sep='\t')
            elif ext in {'.xls', '.xlsx'}:
                df = pd.read_excel(file_path)
            elif ext == '.json':
                df = pd.read_json(file_path)
            elif ext == '.parquet':
                df = pd.read_parquet(file_path)
            else:
                print(f"   [Skipped preview for extension '{ext}']")
                continue
                
            print(f"   * Total Rows: {len(df):,}")
            print(f"   * Columns: {list(df.columns)}")
            print(f"   * Missing / Nulls: {df.isnull().sum().to_dict()}")
            print(f"   * Duplicates: {df.duplicated().sum():,}")
            print(f"   * First 3 Rows:\n{df.head(3)}")
            print()
        except Exception as e:
            print(f"   [!] Could not read '{file_path.name}': {e}\n")

inspect_folder(DATASET_PATH)

### Step 3: Validate the Tourism Knowledge Base (System B)
Checks that your `data/knowledge_base.json` adheres to all schema, safety, and source URL standards.

In [ ]:
from scripts.validate_knowledge_base import validate_kb
from pathlib import Path

kb_path = Path('data/knowledge_base.json')
is_valid = validate_kb(kb_path)
print("Knowledge base valid:", is_valid)

### Step 4: Next Steps for the Team
1. Review `data/phrasebook.json` with a fluent Khasi speaker to verify unverified phrases.
2. Run the Streamlit web application: `streamlit run app.py`
3. Set aside a clean test split before beginning any translation model fine-tuning.